# HF Pipelines

In [ ]:
from tqdm.notebook import tqdm
from transformers import pipeline

We specifying pipeline for certain task. In this case is sentiment analysis.

In [ ]:
classifier = pipeline("sentiment-analysis")

Our, just created pipeline is a classifier which is used for detection if sentence is positive or negative. We can use it like this

In [ ]:
classifier(
    [
    "My name is Rokas. I'm a data engineer from Lithuania.",
    "My hometown is Šiauliai"
    ]
)

## Tokenizer

To convert text into numbers, we use tokenizer.
Each tokenizer has it's vocabulary, which is a mapping of tokens to their corresponding IDs. 
Tokenizer steps:
1. **Tokenization**: The input text is split into smaller units called tokens. It's a parts of the text with some special simbols for separating sentences, paragraphs, etc.
2. **Conversion to IDs**: Each token is then converted into a unique numerical ID based on the tokenizer's vocabulary. This allows the model to process the text as numbers.

In [ ]:
from transformers import AutoTokenizer

We can load tokenizer from pretrained model.

In [ ]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
tokenizer

We can pass text to tokenizer and see created tokens with their corresponding IDs.

In [ ]:
raw_inputs = [
    "My name is Rokas. I'm a data engineer from Lithuania.",
    "My hometown is Šiauliai"
]

inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")

In [ ]:
inputs

## Model

### For hidden state retrieval

Hidden state is the indermediate representation of the input data as it passes through the layers of a neural network. It captures the learned features and patterns from the input data at each layer. Hidden states are crucial for understanding how the model processes and transforms the input data, and they can be used for various downstream tasks such as classification, generation, or feature extraction.

<!-- Add image from assets/hidden.png here -->
![alt](assets/hidden.png)

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(checkpoint)

To get hidden states, we need to specify it when creating pipeline. We can do it like this:

In [ ]:
outputs = model(**inputs)
print(outputs.last_hidden_state.shape)

In [ ]:
outputs

In [ ]:
outputs[0][1][15]

## Model for classification

If we want to use model for classification, we can do use same checkpoint, but we need to specify task when creating pipeline. In this case is "text-classification". We can do it like this

![alt](assets/whole.png)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)


In [ ]:
outputs = model(**inputs)

Now instead of hidden states, we get logits of each sentence, that was defined earlier.

In [ ]:
outputs

In [ ]:
print(outputs.logits.shape)

### Aplly softmax

To make sense of our output results we apply softmax function to put our logits between 0 and 1.

In [ ]:
import torch

predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)


In [ ]:
outputs.logits

First prediction output for negative output, second for positive. In our case both sentences are for second output is closer to 1, so it means they both are positive sentences.

In [ ]:
print(predictions)

We can check if sentence outputs equals to one.

In [ ]:
predictions[0][0] + predictions[0][1]

Cell below show possible output of the model in labels.

In [ ]:
model.config.id2label